# Field Spectra, Airborne AVIRIS and Spaceborne EMIT

## Field spectra
We will use this spectroscopy dataset collected as a part of BioSCAPE project at Greater Cape Floristic Region, South Africa: https://doi.org/10.3334/ORNLDAAC/2482.

In [ ]:
import earthaccess
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from pyproj import Proj, CRS
from os import path
from shapely.ops import orient
import xarray as xr
import rioxarray
from emit_tools import emit_xarray
import warnings
warnings.filterwarnings('ignore')

### Earthdata Authentication
We recommend authenticating your Earthdata Login (EDL) information using the earthaccess python library as follows:

In [ ]:
earthaccess.login(persist=True)

### Search for spectra file
We will use `earthaccess` to search for one of the granule: `BioSCape_foliar_trait_spec.BioSCape_foliar_vnir_spectra.csv`

In [ ]:
doi = '10.3334/ORNLDAAC/2482'
# searching files
granules = earthaccess.search_data(
    doi=doi, # dataset doi
    granule_name = "*spectra.csv",
)
print(f"Total of {len(granules)} found.")

In [ ]:
granules[0]

### Open the spectra file

In [ ]:
for fh in earthaccess.open(granules):
    spectra_df = pd.read_csv(fh)

In [ ]:
spectra_df

### Subset spectra for a species and region

In [ ]:
# plot for Erica breviflora
sc_name = "Erica breviflora"
spectra_df = spectra_df[spectra_df.scientific_name_original == sc_name]
spectra_df

In [ ]:
subregion = "hangklip"
spectra_df = spectra_df[spectra_df.subregion == subregion]
spectra_df

### Plot spectral plots

In [ ]:
xyz = "https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}"
attr = "ESRI"
spectra_gdf = gpd.GeoDataFrame(spectra_df, crs="EPSG:4326",
                       geometry=gpd.points_from_xy(spectra_df["longitude"],
                                                   spectra_df["latitude"]))
m = spectra_gdf.explore(tiles=xyz, attr=attr)
m

### Plot spectra

In [ ]:

# Find spectral columns
x_columns = sorted(
    [col for col in spectra_df.columns if col.startswith("X")],
    key=lambda x: float(x[1:])
)

# create pandas
spectra_arr = []

for idx, row in spectra_df.iterrows():
    for col in x_columns:
        spectra_arr.append({
            "spectrum_id": f"spectra_{idx}",
            "wavelength": float(col[1:]),  # X500 -> 500
            "reflectance": row[col]/100,
            "source": "Field Spectra"
        })

all_df = pd.DataFrame(spectra_arr)

plt.figure(figsize=(14, 8))

for spectrum_id, group in all_df.groupby("spectrum_id"):
    plt.plot(
        group["wavelength"],
        group["reflectance"],
        alpha=0.7,
        label=spectrum_id
    )

plt.xlabel("Wavelength (nm)")
plt.ylabel("Value")
plt.title("Spectral Curves")

# Hide legend if there are too many spectra
if all_df["spectrum_id"].nunique() <= 20:
    plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")

plt.tight_layout()
plt.show()

## AVIRIS-NG
We’ll use this bounding boxs to refine our search of BioSCape AVIRIS-NG L3 files in our region.

Again, using earthaccess we can query the data to discover files from the dataset: [BioSCape_ANG_V02_L3_RFL_Mosaic_2427, BioSCape: AVIRIS-NG L3 Resampled Reflectance Mosaics, V2 ](https://www.earthdata.nasa.gov/data/catalog/ornl-cloud-bioscape-ang-v02-l3-rfl-mosaic-2427-2).

### Search AVIRIS tile

In [ ]:
lon = spectra_gdf.longitude.iloc[0]
lat = spectra_gdf.latitude.iloc[0]
granules = earthaccess.search_data(
    short_name = 'BioSCape_ANG_V02_L3_RFL_Mosaic_2427', 
    point=(lon, lat),
    granule_name=('*AVIRIS-NG_BIOSCAPE_V02_L3*')
)
print(f"Total granules found: {len(granules)}")

### Map the AVIRIS boundary

In [ ]:
aviris_gdf = gpd.GeoDataFrame(granules, geometry=gpd.GeoSeries(granules, crs=4326))
m = aviris_gdf.explore(m=m, tiles=xyz, attr=attr)
m

In [ ]:
granules[0]

### Open AVIRIS reflectance file

In [ ]:
def get_s3_links(g, suffix_str):
    return [i for i in g.data_links(access="direct") if i.endswith(suffix_str)][0]

rfl_f = []
for g in granules:
    rfl_f.append(get_s3_links(g, 'RFL.nc'))
rfl_f

In [ ]:
s3_f = earthaccess.open(rfl_f, provider="ORNL_CLOUD")[0]
ds = xr.open_dataset(s3_f, engine="h5netcdf", decode_coords=all)
ds

### Extracting spectra for the location

In [ ]:
# converting lon, lat to AVIRIS CRS
p = Proj(ds.transverse_mercator.crs_wkt)
x, y = p(lon, lat)
x, y

In [ ]:
rfl_ds = xr.open_datatree(s3_f, engine='h5netcdf').reflectance.to_dataset()
rfl_ds

In [ ]:
spec_ds = rfl_ds.reflectance.sel(easting=x, northing=y, method='nearest')
spec_ds

### Plot the spectra

In [ ]:
spec_df = spec_ds.to_dataframe().reset_index()[["wavelength", "reflectance"]]
spec_df["spectrum_id"] = "AVIRIS"
spec_df["source"]= "AVIRIS"
# all_df = pd.concat([all_df, spec_df], ignore_index=True)

plt.figure(figsize=(14, 8))

for spectrum_id, group in spec_df.groupby("spectrum_id"):
    plt.plot(
        group["wavelength"],
        group["reflectance"],
        alpha=0.7,
        label=spectrum_id
    )

plt.xlabel("Wavelength (nm)")
plt.ylabel("Value")
plt.title("Spectral Curves")

# Hide legend if there are too many spectra
if spec_df["spectrum_id"].nunique() <= 20:
    plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")

plt.tight_layout()
plt.show()

## EMIT L2A Spectra

In [ ]:
granules = earthaccess.search_data(
    short_name = 'EMITL2ARFL', 
    temporal = ("2023-01-01", "2023-12-30"),
    point=(lon, lat)
)
print(f"Total granules found: {len(granules)}")

In [ ]:
emit_gdf = gpd.GeoDataFrame(granules, geometry=gpd.GeoSeries(granules, crs=4326))
emit_gdf.explore(m=m, color="red", tiles=xyz, attr=attr)

In [ ]:
granules[0]

In [ ]:
def get_s3_links(g, suffix_str):
    return [i for i in g.data_links(access="direct") if path.basename(i).startswith(suffix_str)][0]

rfl_f = []
for gr in granules:
    rfl_f.append(get_s3_links(gr, 'EMIT_L2A_RFL'))
rfl_f

In [ ]:
s3_emit = earthaccess.open(rfl_f[-1:], provider="LPCLOUD")[0]

In [ ]:
ds_geo = emit_xarray(s3_emit, ortho=True)
ds_geo

In [ ]:
point = ds_geo.sel(longitude=lon,latitude=lat,method='nearest')